# Simple PDF RAG — Clean End-to-End Demo Notebook (Ingestion → Retrieval → Response)

It shows a complete, runnable pipeline:

## What we will do
1. Load a PDF (or multiple PDFs)
2. Split into chunks
3. Embed chunks using a local embedding model - sentence-transformers/all-MiniLM-L6-v2
4. Store embeddings in a persistent vector DB - Chroma
5. Retrieve the most relevant chunks for a question
6. Generate a grounded answer using retrieved context with local LLM - Ollama 

## Deliverables
-  Persistent vector store created at: `db/chroma_pdf/`
-  Visible chunk counts + chunk previews
-  Visible retrieved chunks (with page number metadata)
-  Final answer generated from **retrieved context only**


In [1]:
!pip install -U \
langchain \
langchain-community \
langchain-huggingface \
langchain-chroma \
langchain-ollama \
langchain-text-splitters \
chromadb \
sentence-transformers \
pypdf



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import textwrap
from pathlib import Path
from typing import List

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Local LLM (Ollama)
from langchain_ollama import ChatOllama


c:\Users\kavya parige\OneDrive\Desktop\Final_rag_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---- Files ----
PDF_PATH = "data/book.pdf"     
PERSIST_DIR = "db/chroma_pdf"
COLLECTION_NAME = "pdf_book"

# ---- Models ----
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OLLAMA_MODEL = "phi3:mini"     # can be llama3.2, mistral, etc.

# ---- Chunking ----
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 0

# ---- Retrieval ----
TOP_K = 3
USE_THRESHOLD = False
SCORE_THRESHOLD = 0.3


In [5]:
print("PDF exists?", os.path.exists(PDF_PATH))
print("Persist dir exists?", os.path.exists(PERSIST_DIR))
print("PDF path:", PDF_PATH)
print("Persist dir:", PERSIST_DIR)


PDF exists? True
Persist dir exists? False
PDF path: data/book.pdf
Persist dir: db/chroma_pdf


In [6]:
def load_pdf(pdf_path: str):
    print(f"Loading PDF: {pdf_path}")
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")


    # Load all pages from the docs directory
    docs = PyPDFLoader(pdf_path).load()
    print(f"Loaded pages: {len(docs)}")
    return docs

docs = load_pdf(PDF_PATH)

Loading PDF: data/book.pdf
Loaded pages: 1208


In [9]:
def split_docs(docs, chunk_size=1000, chunk_overlap=0):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    print(f"Total chunks created: {len(chunks)}")

    # Show 3 chunk previews (interview-friendly)
    for i, c in enumerate(chunks[:3], 1):
        print(f"\n--- Chunk {i} ---")
        print("Page:", c.metadata.get("page", "NA"))
        print(textwrap.fill(c.page_content[:450], width=110))
        print("-" * 70)

    return chunks

chunks = split_docs(docs, CHUNK_SIZE, CHUNK_OVERLAP)


Total chunks created: 1990

--- Chunk 1 ---
Page: 0
Human Nutrition: 2020 Edition
----------------------------------------------------------------------

--- Chunk 2 ---
Page: 2
Human Nutrition: 2020 Edition UNIVER SITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN
TIT CHENAL, SKYLAR HARA, NOEMI ARCEO CAA CBAY, WILLIAM MEINKE-LA U, YA-YUN YANG, MARIE KAINO A FIALK OWSKI
REVILLA, JENNIFER DRAPER, GEMAD Y LANGFELDER, CHER YL GIBBY , CHYNA NICOLE CHUN, AND ALLISON CALABRESE
----------------------------------------------------------------------

--- Chunk 3 ---
Page: 3
Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is
licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.
----------------------------------------------------------------------


In [10]:
def get_db(persist_dir: str, collection_name: str):
    embedding_model = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

    db = Chroma(
        persist_directory=persist_dir,
        collection_name=collection_name,
        embedding_function=embedding_model
    )
    return db


def build_or_load_vectorstore(chunks):
    # If DB exists, load it (fast interview demo)
    if os.path.exists(PERSIST_DIR):
        print(f"Vector store already exists. Loading from: {PERSIST_DIR}")
        db = get_db(PERSIST_DIR, COLLECTION_NAME)
        try:
            print("Stored chunks (count):", db._collection.count())
        except Exception:
            print("Loaded DB (count not available).")
        return db

    # Else create and persist it
    print("No vector store found. Creating a new one...")
    embedding_model = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=PERSIST_DIR,
        collection_name=COLLECTION_NAME
    )
    print(f"Vector store created at: {PERSIST_DIR}")
    return db

db = build_or_load_vectorstore(chunks)


No vector store found. Creating a new one...
Vector store created at: db/chroma_pdf


In [15]:
def retrieve_chunks(db, query: str, k: int = 3, use_threshold: bool = False, score_threshold: float = 0.3):
    if use_threshold:
        retriever = db.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={"k": k, "score_threshold": score_threshold}
        )
    else:
        retriever = db.as_retriever(search_kwargs={"k": k})

    docs = retriever.invoke(query)

    print(f"\nUser Query: {query}")
    print("\n--- Retrieved Context ---\n")

    if not docs:
        print("No documents retrieved.")
        return []

    for i, d in enumerate(docs, 1):
        print(f"Chunk {i} | Page: {d.metadata.get('page','NA')}")
        print(textwrap.fill(d.page_content[:800], width=110))
        print("-" * 90)

    return docs


QUERY = "wt is fiber?"
retrieved_docs = retrieve_chunks(
    db=db,
    query=QUERY,
    k=TOP_K,
    use_threshold=USE_THRESHOLD,
    score_threshold=SCORE_THRESHOLD
)



User Query: wt is fiber?

--- Retrieved Context ---

Chunk 1 | Page: 278
Dietary fiber is c ategorized as ei ther water-soluble or insoluble. Some e xamples o f soluble f ibers ar e
inulin, pe ctin, and guar gum and they are found in pe as, beans, oats, barley, and r ye. Cellulose and lignin
ar e insoluble f ibers and a f ew die tary sour ces o f them are whole-grain foods, f lax, cauliflower, and a
vocados. Cellulose is the most abundan t f iber in plan ts, making up the c ell w alls and providing struc
ture. Soluble f ibers ar e mor e e asily ac cessible to bacterial enzymes in the large intestine so they can
be broken down to a greater extent than insoluble fibers, but even some breakdown of cellulose and other
insoluble fibers occurs. The last class o f f iber is func tional f iber. Func tional f ibers have been adde d
to f oods and ha ve be en sho wn to pr
------------------------------------------------------------------------------------------
Chunk 2 | Page: 278
is psy llium-s

In [16]:
def build_prompt(question: str, docs) -> str:
    context_blocks = []
    for d in docs:
        page = d.metadata.get("page", "NA")
        context_blocks.append(f"[Page {page}] {d.page_content}")

    context = "\n\n".join(context_blocks)

    prompt = f"""
You are a helpful assistant.\n
Answer the question using ONLY the provided context.\n
If the answer is not present in the context,\n
say exactly: NOT AVAILABLE IN THE PROVIDED PDF CONTEXT.

\n QUESTION: {question}

\n CONTEXT: {context}

ANSWER:
""".strip()

    return prompt


if retrieved_docs:
    prompt = build_prompt(QUERY, retrieved_docs)
    print("\n Prompt preview (first 600 chars):\n")
    print(prompt[:600])
else:
    print("No retrieved docs → skipping prompt build.")



 Prompt preview (first 600 chars):

You are a helpful assistant.

Answer the question using ONLY the provided context.

If the answer is not present in the context,

say exactly: NOT AVAILABLE IN THE PROVIDED PDF CONTEXT.


 QUESTION: wt is fiber?


 CONTEXT: [Page 278] Dietary fiber is c ategorized as ei ther water-soluble or insoluble.
Some e xamples o f soluble f ibers ar e inulin, pe ctin, and guar gum
and they are found in pe as, beans, oats, barley, and r ye. Cellulose
and lignin ar e insoluble f ibers and a f ew die tary sour ces o f them
are whole-grain foods, f lax, cauliflower, and a vocados. Cellulose is
the most abun


In [17]:
def generate_answer(prompt: str, model_name: str = "phi3:mini", temperature: float = 0.0):
    llm = ChatOllama(model=model_name, temperature=temperature)
    response = llm.invoke(prompt).content
    return response


if retrieved_docs:
    answer = generate_answer(prompt, model_name=OLLAMA_MODEL, temperature=0.0)
    print("\nFinal Answer\n")
    print(answer)
else:
    print("No retrieved docs → cannot generate grounded answer.")



Final Answer

Fiber, specifically dietary fiber and functional fiber, can be found in various foods such as whole-grain products (e.g., bran), vegetables like cauliflower, fruits, legumes including peas and beans, seeds from flaxseeds to sesame seeds, nuts, nut butters, soybeans, oats, barley, rye, psyllium-seed husk (a functional fiber), as well as in foods like popcorn. Soluble fibers are found in peas and beans while insoluble fibers can also come from whole grains among other sources mentioned above.
